# **Obtención de resultados OCR de tres modelos sobre un dataset**

Se realiza OCR sobre las imágenes de un dataset de dorsales y se guarda en un archivo CSV por cada una, qué texto, confianza, y tiempo de procesamiento usa cada OCR (tesseract, easyocr, y Parseq). También guarda el split del dataset, el nombre de la imagen, y el número real que se ve en el dorsal.

Se siguen los siguientes pasos:
1. **Librerías y clases**: Se importan librerías y clases necesarias para ejecutar el notebook
2. **Creación del dataframe**: Se crea un dataframe para guardar las imágenes y los resultados que da cada OCR
3. **Definición de clases de los tres OCRs**: Se definen clases para obtener el texto, confianza, y tiempo de procesamiento de cada OCR en una imagen de un dorsal
4. **Obtención de los resultados de OCR**: Se obtiene el texto, confianza, y tiempo de procesamiento por cada imagen del conjunto de datos utilizado y se guarda en el dataframe anteriormente creado
5. **Almacenar los resultados de realizar OCR**: Se almacenan los resultados en un CSV

## **Librerías y clases**

In [ ]:
#@title Añadir librerías
import pandas as pd

import torch
from tqdm.notebook import tqdm

# IoU para comparar bboxes
from torchvision.ops import box_iou

import cv2

import os
import glob

# Obtener clases propias en subcarpetas
import sys
from pathlib import Path

In [ ]:
#@title Añadir clases de código propias

# Subcarpetas con clases que se utilizan en este notebook
rutas = ["MejoraSeguimiento", "Auxiliares", "VisionArtificial"]

# Se añaden las rutas al path para poder importar las clases
for ruta in rutas:
     ruta_completa = str(Path.cwd() / ruta)
     if ruta_completa not in sys.path:
         sys.path.append(ruta_completa)

# Se importan las clases necesarias para el notebook
from Detector import Detector
from OCR import OCR

Inicializar clases que después se usarán

In [ ]:
#@title Inicializar el detector y el OCR
device = 'cuda' if torch.cuda.is_available() else 'cpu'
detector = Detector(device=device)
ocr = OCR(device=device)

Indicar dónde se encuentra cada split. Es necesario descargar en estas carpetas el dataset antes de ejecutar este notebook.

In [ ]:
DATA_PATH_TEST = "./data/OCR/test"
DATA_PATH_VALID = "./data/OCR/valid"
DATA_PATH_TRAIN = "./data/OCR/train"

## **Creación del dataframe**

Se crea el dataframe únicamente con la ruta a cada imagen, el split al que pertenece, y el número que contiene. Tras crearlo, se añaden las columnas para los resultados del texto predicho por cada OCR, la confianza, y el tiempo que tarda.

In [ ]:
import json
import pandas as pd
import numpy as np

# Lee el archivo jsonl y extrae los campos importantes para crear un DataFrame
datos_procesados = []

# Test
jsonl_file_path = f"{DATA_PATH_TEST}/annotations.jsonl"
with open(jsonl_file_path, 'r', encoding='utf-8') as archivo:
    for linea in archivo:
        # Cada línea es un string JSON independiente
        registro = json.loads(linea)
        
        # Guardamos la imagen, el número al que se refiere y su split (test, valid o train)
        datos_procesados.append({
            'imagen': registro['image'],
            'numero_real': registro['suffix'],
            "split": "test"
        })
        
# Valid (lo mismo que en test, pero con el split "valid")
jsonl_file_path = f"{DATA_PATH_VALID}/annotations.jsonl"
with open(jsonl_file_path, 'r', encoding='utf-8') as archivo:
    for linea in archivo:
        registro = json.loads(linea)
        
        datos_procesados.append({
            'imagen': registro['image'],
            'numero_real': registro['suffix'],
            "split": "valid"
        })

# Train (lo mismo que en test, pero con el split "train")
jsonl_file_path = f"{DATA_PATH_TRAIN}/annotations.jsonl"
with open(jsonl_file_path, 'r', encoding='utf-8') as archivo:
    for linea in archivo:
        registro = json.loads(linea)
        
        datos_procesados.append({
            'imagen': registro['image'],
            'numero_real': registro['suffix'],
            "split": "train"
        })


# Crea el DataFrame inicial
df = pd.DataFrame(datos_procesados)

# Se inicializan las columnas para los resultados de los OCRs
df['tesseract_texto'] = None
df['tesseract_confianza'] = None
df['tesseract_tiempo'] = None

df['easyocr_texto'] = None
df['easyocr_confianza'] = None
df['easyocr_tiempo'] = None

df['parseq_texto'] = None
df['parseq_confianza'] = None
df['parseq_tiempo'] = None

# Ver el resultado de las primeras filas
df.head()

## **Definición de clases de los tres OCRs**

Se crean 3 clases, una por cada OCR. Cada clase tiene el método predict en el que procesa una imagen y devuelve el texto leído, la confianza, y el tiempo que tarda en procesar dicha imagen.

In [ ]:
# Clase base para los modelos OCR
class OCRModel:
    def __init__(self, name):
        self.name = name
    
    def predict(self, image_path):
        # Por implementar en subclases
        raise NotImplementedError("Las subclases deben implementar este método")
    
import pytesseract
from PIL import Image
# Cambiar según la ruta de instalación. Es necesario que Tesseract esté instalado en el sistema y que la ruta sea correcta.
pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\TesseractOCR\tesseract.exe'
import time
class TesseractOCR(OCRModel):
    def __init__(self):
        super().__init__("tesseract")
    
    def predict(self, image_path):
        """Devuelve el texto reconocido, la confianza, y el tiempo de procesamiento"""
        start_time = time.time()
        diccionario = pytesseract.image_to_data(image_path, output_type=pytesseract.Output.DICT)
        texto = diccionario['text']
        confianza = diccionario['conf']
        end_time = time.time()
        tiempo_procesamiento = end_time - start_time
        return texto, confianza, tiempo_procesamiento
    
import easyocr
class EasyOCRModel(OCRModel):
    def __init__(self, device):
        super().__init__("easyocr")
        self.reader = easyocr.Reader(['en'], gpu=(device=='cuda'))
    
    def predict(self, image_path):
        """Devuelve el texto reconocido, la confianza, y el tiempo de procesamiento"""
        start_time = time.time()
        img_array = cv2.imread(image_path)
        resultado = self.reader.readtext(image=img_array)
        texto = [res[1] for res in resultado]
        confianza = [res[2] for res in resultado]
        end_time = time.time()
        tiempo_procesamiento = end_time - start_time
        return texto, confianza, tiempo_procesamiento
    

class PARSEQModel(OCRModel):
    def __init__(self, device):
        super().__init__("parseq")
        self.parseq = torch.hub.load('baudm/parseq', 'parseq', pretrained=True, trust_repo=True).eval()
        self.parseq = self.parseq.to(device)
        self.device = device
        # Si se importa antes la librería da error por no encontrar baudm/parseq, por lo que se importa después de obtener el modelo
        from strhub.data.module import SceneTextDataModule
        self.img_transform = SceneTextDataModule.get_transform(self.parseq.hparams.img_size)
    def predict(self, image_path):
        """Devuelve el texto reconocido, la confianza, y el tiempo de procesamiento"""
        start_time = time.time()
        img = Image.open(image_path).convert('RGB')
        img_tensor = self.img_transform(img).unsqueeze(0).to(self.device)
        with torch.no_grad():
            output = self.parseq(img_tensor)
            prediccion = output.softmax(-1)
            label, confidence = self.parseq.tokenizer.decode(prediccion)
            texto = label[0]
            confianza = confidence[0]
        end_time = time.time()
        tiempo_procesamiento = end_time - start_time
        return texto, confianza, tiempo_procesamiento

## **Obtención de los resultados de OCR**

Para cada OCR guarda los resultados de procesar cada imagen del conjunto de datos.

In [ ]:
tesseract_ocr = TesseractOCR()

for index, row in tqdm(df.iterrows(), total=len(df)):
    data_path = None
    if row['split'] == "test":
        data_path = DATA_PATH_TEST
    elif row['split'] == "valid":
        data_path = DATA_PATH_VALID
    elif row['split'] == "train":
        data_path = DATA_PATH_TRAIN
    imagen_path = os.path.join(data_path, row['imagen'])
    texto, confianza, tiempo = tesseract_ocr.predict(imagen_path)
    df.at[index, 'tesseract_texto'] = texto
    df.at[index, 'tesseract_confianza'] = confianza
    df.at[index, 'tesseract_tiempo'] = tiempo

In [ ]:
easyocr_model = EasyOCRModel(device=device)
for index, row in tqdm(df.iterrows(), total=len(df)):
    data_path = None
    if row['split'] == "test":
        data_path = DATA_PATH_TEST
    elif row['split'] == "valid":
        data_path = DATA_PATH_VALID
    elif row['split'] == "train":
        data_path = DATA_PATH_TRAIN
    imagen_path = os.path.join(data_path, row['imagen'])
    texto, confianza, tiempo = easyocr_model.predict(imagen_path)
    df.at[index, 'easyocr_texto'] = texto
    df.at[index, 'easyocr_confianza'] = confianza
    df.at[index, 'easyocr_tiempo'] = tiempo

In [ ]:
# Parseq OCR
parseq_model = PARSEQModel(device=device)
for index, row in tqdm(df.iterrows(), total=len(df)):
    data_path = None
    if row['split'] == "test":
        data_path = DATA_PATH_TEST
    elif row['split'] == "valid":
        data_path = DATA_PATH_VALID
    elif row['split'] == "train":
        data_path = DATA_PATH_TRAIN
    imagen_path = os.path.join(data_path, row['imagen'])
    texto, confianza, tiempo = parseq_model.predict(imagen_path)
    df.at[index, 'parseq_texto'] = texto
    df.at[index, 'parseq_confianza'] = confianza
    df.at[index, 'parseq_tiempo'] = tiempo

## **Almacenar los resultados de realizar OCR**

Almacena los resultados (el dataframe creado) en un fichero CSV

In [ ]:
# Almacenar resultados en CSV
df.to_csv(f"./resultados_ocr.csv", index=False)

In [ ]:
# Volver a cargar el CSV para mostrarlo
df_resultados = pd.read_csv(f"./resultados_ocr.csv")
df_resultados.head()